# 02 — Data Cleaning & ETL Pipeline
**NST DVA Capstone 2 | B_G6_DIGITOMICS**

**Sector:** EdTech / Digital Behaviour Analytics  
**Dataset:** Student Digital Behaviour Data  
**Notebook Purpose:** Apply a fully documented, step-by-step Python cleaning pipeline to produce a standardised, analysis-ready dataset saved to `data/processed/`.

---
**Pipeline Steps in this notebook:**
1. Load raw data  
2. Drop irrelevant columns  
3. Handle missing values  
4. Standardise string / categorical values  
5. Encode categorical columns  
6. Fix data types  
7. Detect and treat outliers  
8. Feature engineering (derived KPI columns)  
9. Final validation  
10. Save cleaned dataset

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Display all columns in output
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded.')

Libraries loaded.


## 2. Load Raw Dataset

In [2]:
RAW_PATH       = '../data/raw/student_digital_behaviour_data.csv'
PROCESSED_PATH = '../data/processed/student_digital_behaviour_cleaned.csv'

# Create processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv(RAW_PATH)

print(f'Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Raw dataset loaded: 500,000 rows × 48 columns


,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,device_access,internet_access_hours,education_level,field_of_study,academic_motivation,online_learning_hours,social_media_hours,sessions_per_day,average_session_length_minutes,late_night_usage,education_content_hours,short_video_hours,entertainment_content_hours,news_content_hours,likes_given_per_day,comments_written_per_day,posts_created_per_week,late_night_score,brain_rot_index,brain_rot_level,attention_span_minutes,study_hours_per_week,class_attendance_rate,productivity_score,sleep_hours,stress_level,anxiety_score,depression_score,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Qatar,Developing,16.3000,54.9300,27.1900,21,Male,Rural,High,Smartphone,5.2395,Graduate,Business,7.1772,10.4244,1.0987,4.6488,16.0574,Sometimes,0.2399,0.4570,0.4018,0.3294,30.0963,4.9677,2.1084,1,5.8685,Low,57.5522,31.4342,95.4824,10.0000,7.1390,4.2620,3.1929,1.4091,38.6995,3.1196,3.6811,79.5082,No,No,8.1799,66.6622,0.0000,26.5167
1,2,USA,Developed,8.7500,94.3900,85.3400,25,Male,Urban,Middle,Smartphone,6.2624,Diploma,Arts,5.4917,8.1464,5.6115,15.1404,63.0832,Often,0.3832,3.3061,1.9222,0.6437,114.3857,3.1745,5.4780,2,29.5948,Low,51.3099,25.9784,86.6170,7.1794,4.0000,8.2135,8.0649,7.6952,157.4004,18.3581,6.5389,73.4525,No,Yes,22.0731,43.8923,0.0000,39.2577
2,3,Mexico,Developing,23.6400,47.2400,73.5500,18,Female,Urban,Low,Smartphone,3.4377,Dropout,NaN,5.3950,5.1545,2.1999,7.0752,25.7789,Sometimes,0.3954,0.8847,0.9198,0.4122,56.3332,8.2406,1.1245,1,20.2423,Low,55.8444,21.4475,89.6282,9.5730,6.8093,2.9313,1.7192,1.0000,79.6035,11.7583,5.1507,35.7531,No,No,12.5918,65.4846,0.0000,47.5422


---
## STEP 1 — Drop Irrelevant Columns

**Rationale:** `student_id` is a row identifier only — it carries no analytical value and should not be treated as a numeric feature. We preserve it as an index.

In [3]:
# Log shape before
print(f'Before: {df.shape}')

# Set student_id as index (preserves the ID for traceability but excludes from analysis)
df = df.set_index('student_id')

print(f'After (student_id moved to index): {df.shape}')
print('→ STEP 1 complete: student_id set as index.')

Before: (500000, 48)
After (student_id moved to index): (500000, 47)
→ STEP 1 complete: student_id set as index.


---
## STEP 2 — Handle Missing Values

**Rationale:**
- `field_of_study`: NaN for students at school or dropouts who have no declared major. Imputing with `'Not Applicable'` preserves real meaning.
- All other NaN values are checked and handled per column logic.

In [4]:
# --- Audit missing values before treatment ---
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]
print('Missing values BEFORE treatment:')
print(missing_before.to_string() if not missing_before.empty else 'None detected.')

Missing values BEFORE treatment:
field_of_study     247349
brain_rot_level      6262


In [5]:
# --- field_of_study: NaN → 'Not Applicable' ---
# Reason: School students and dropouts have no declared field. NaN is structurally correct
# but must be filled to allow groupBy operations in analysis and Tableau.
if 'field_of_study' in df.columns:
    n_filled = df['field_of_study'].isnull().sum()
    df['field_of_study'] = df['field_of_study'].fillna('Not Applicable')
    print(f'field_of_study: filled {n_filled} NaN values → "Not Applicable"')

field_of_study: filled 247349 NaN values → "Not Applicable"


In [6]:
# --- Numeric columns: fill remaining NaN with column median ---
# Reason: Median is robust to skew/outliers. Use only for columns with <5% missing.
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    n_null = df[col].isnull().sum()
    pct_null = n_null / len(df) * 100
    if n_null > 0:
        if pct_null <= 5:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f'{col}: filled {n_null} NaN ({pct_null:.1f}%) with median ({median_val:.4f})')
        else:
            print(f'⚠️  {col}: {pct_null:.1f}% missing — review manually before imputing')

In [7]:
# --- Confirm: No remaining NaN ---
missing_after = df.isnull().sum().sum()
print(f'Total missing values AFTER treatment: {missing_after}')
print('→ STEP 2 complete: Missing values handled.')

Total missing values AFTER treatment: 6262
→ STEP 2 complete: Missing values handled.


---
## STEP 3 — Standardise String / Categorical Values

**Rationale:** Free-text inconsistencies (extra spaces, mixed case, trailing whitespace) create false duplicates in `groupby` and Tableau filters. Strip and title-case all object columns.

In [8]:
str_cols = df.select_dtypes(include='object').columns

for col in str_cols:
    before_unique = df[col].nunique()
    df[col] = df[col].str.strip().str.title()
    after_unique = df[col].nunique()
    if before_unique != after_unique:
        print(f'{col}: {before_unique} → {after_unique} unique values after strip/title (duplicates merged)')
    else:
        print(f'{col}: {after_unique} unique values — no change in cardinality')

print()
print('→ STEP 3 complete: String standardisation applied.')

country: 50 unique values — no change in cardinality
development_level: 3 unique values — no change in cardinality
gender: 3 unique values — no change in cardinality
urban_rural: 2 unique values — no change in cardinality
family_income_level: 3 unique values — no change in cardinality
device_access: 4 unique values — no change in cardinality
education_level: 6 unique values — no change in cardinality
field_of_study: 7 unique values — no change in cardinality
late_night_usage: 4 unique values — no change in cardinality
brain_rot_level: 2 unique values — no change in cardinality
cyberbullying_exposure: 2 unique values — no change in cardinality
adult_content_exposure: 2 unique values — no change in cardinality

→ STEP 3 complete: String standardisation applied.


---
## STEP 4 — Encode Categorical Columns

**Encoding strategy:**

| Column | Type | Encoding |
|--------|------|----------|
| `late_night_usage` | Ordinal | `Never=0, Sometimes=1, Often=2` |
| `cyberbullying_exposure` | Binary | `No=0, Yes=1` |
| `adult_content_exposure` | Binary | `No=0, Yes=1` |
| `brain_rot_level` | Ordinal | `Low=1, Medium=2, High=3` |
| `gender`, `country`, `urban_rural`, etc. | Nominal | Kept as string for Tableau; numeric copy added only where needed |

In [9]:
# --- late_night_usage: Ordinal encoding ---
late_night_map = {'Never': 0, 'Sometimes': 1, 'Often': 2}

if 'late_night_usage' in df.columns:
    # Check for unexpected values before mapping
    unexpected = set(df['late_night_usage'].unique()) - set(late_night_map.keys())
    if unexpected:
        print(f'⚠️  Unexpected values in late_night_usage: {unexpected}')
    
    df['late_night_usage_encoded'] = df['late_night_usage'].map(late_night_map)
    null_encoded = df['late_night_usage_encoded'].isnull().sum()
    print(f'late_night_usage_encoded created. Nulls after mapping: {null_encoded}')
    print(df[['late_night_usage', 'late_night_usage_encoded']].value_counts().reset_index().to_string(index=False))

⚠️  Unexpected values in late_night_usage: {'Always'}
late_night_usage_encoded created. Nulls after mapping: 29784
late_night_usage  late_night_usage_encoded  count
       Sometimes                    1.0000 228653
           Often                    2.0000 123307
           Never                    0.0000 118256


In [10]:
# --- Binary encoding: Yes/No columns ---
binary_cols = ['cyberbullying_exposure', 'adult_content_exposure']

for col in binary_cols:
    if col in df.columns:
        df[f'{col}_flag'] = df[col].map({'Yes': 1, 'No': 0})
        print(f'{col}_flag created → No=0, Yes=1')
        print(f'  Distribution: {df[f"{col}_flag"].value_counts().to_dict()}')

cyberbullying_exposure_flag created → No=0, Yes=1
  Distribution: {0: 438002, 1: 61998}
adult_content_exposure_flag created → No=0, Yes=1
  Distribution: {0: 408125, 1: 91875}


In [11]:
# --- brain_rot_level: Ordinal encoding ---
brain_rot_map = {'Low': 1, 'Medium': 2, 'High': 3}

if 'brain_rot_level' in df.columns:
    unexpected_br = set(df['brain_rot_level'].unique()) - set(brain_rot_map.keys())
    if unexpected_br:
        print(f'⚠️  Unexpected values in brain_rot_level: {unexpected_br}')
    
    df['brain_rot_level_encoded'] = df['brain_rot_level'].map(brain_rot_map)
    print(f'brain_rot_level_encoded created.')
    print(df['brain_rot_level'].value_counts().to_string())

print()
print('→ STEP 4 complete: Categorical encoding applied.')

⚠️  Unexpected values in brain_rot_level: {'Moderate', nan}
brain_rot_level_encoded created.
brain_rot_level
Low         432156
Moderate     61582

→ STEP 4 complete: Categorical encoding applied.


---
## STEP 5 — Fix Data Types

**Rationale:** Ensure numeric columns are float/int and categorical columns are `category` dtype to reduce memory and enable correct Pandas operations.

In [12]:
# Columns that should be categorical dtype (low cardinality, non-numeric)
cat_dtype_cols = [
    'country', 'development_level', 'gender', 'urban_rural',
    'family_income_level', 'device_access', 'education_level',
    'field_of_study', 'late_night_usage', 'brain_rot_level',
    'cyberbullying_exposure', 'adult_content_exposure'
]

for col in cat_dtype_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Ensure age is integer
if 'age' in df.columns:
    df['age'] = df['age'].astype(int)

# Confirm
print('Updated dtypes for key columns:')
print(df.dtypes[cat_dtype_cols].to_string())
print()
print('→ STEP 5 complete: Data types corrected.')

Updated dtypes for key columns:
country                   category
development_level         category
gender                    category
urban_rural               category
family_income_level       category
device_access             category
education_level           category
field_of_study            category
late_night_usage          category
brain_rot_level           category
cyberbullying_exposure    category
adult_content_exposure    category

→ STEP 5 complete: Data types corrected.


---
## STEP 6 — Outlier Detection & Treatment

**Method:** IQR-based capping (Winsorisation) for key numeric KPI columns.  
**Rationale:** Hard removal deletes real students. Capping at 1.5× IQR fence preserves row count while controlling extreme influence on statistics.  
**Columns treated:** High-impact numeric features used in KPIs and regression.

In [13]:
# Columns to inspect for outliers
outlier_cols = [
    'online_learning_hours', 'social_media_hours', 'internet_access_hours',
    'study_hours_per_week', 'sleep_hours', 'stress_level', 'anxiety_score',
    'depression_score', 'digital_spending_per_month', 'ads_viewed_per_day',
    'digital_addiction_score', 'wellbeing_index', 'productivity_score'
]

outlier_summary = []
for col in outlier_cols:
    if col in df.columns:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        n_outliers  = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
        outlier_summary.append({
            'Column': col,
            'Q1': round(Q1, 2),
            'Q3': round(Q3, 2),
            'Lower Fence': round(lower_fence, 2),
            'Upper Fence': round(upper_fence, 2),
            'Outlier Count': n_outliers
        })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

                    Column      Q1       Q3  Lower Fence  Upper Fence  Outlier Count
     online_learning_hours  5.0800   7.5900       1.3200      11.3400           3004
        social_media_hours  2.4600   4.2000      -0.1500       6.8000           3639
     internet_access_hours  3.9100   6.1100       0.6100       9.4100           1840
      study_hours_per_week 19.4500  26.6400       8.6700      37.4200           2360
               sleep_hours  5.3100   6.6700       3.2800       8.7000            788
              stress_level  3.8200   5.8600       0.7600       8.9200           1580
             anxiety_score  3.6300   6.0600      -0.0200       9.7200           1734
          depression_score  3.6100   6.5100      -0.7400      10.8600              0
digital_spending_per_month 38.2000  76.7300     -19.5900     134.5300           9567
        ads_viewed_per_day 73.8200 118.6400       6.5800     185.8900           3765
   digital_addiction_score  9.9100  18.0400      -2.3000      30.

In [14]:
# --- Winsorisation: Cap at IQR fences ---
capped_log = []
for col in outlier_cols:
    if col in df.columns:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        
        n_below = (df[col] < lower_fence).sum()
        n_above = (df[col] > upper_fence).sum()
        
        df[col] = df[col].clip(lower=lower_fence, upper=upper_fence)
        
        if n_below + n_above > 0:
            capped_log.append(f'{col}: capped {n_below} below, {n_above} above')

if capped_log:
    print('Winsorisation applied:')
    for log in capped_log:
        print(f'  {log}')
else:
    print('No values required capping (all within IQR fences).')

print()
print('→ STEP 6 complete: Outlier treatment applied.')

Winsorisation applied:
  online_learning_hours: capped 1896 below, 1108 above
  social_media_hours: capped 0 below, 3639 above
  internet_access_hours: capped 949 below, 891 above
  study_hours_per_week: capped 1599 below, 761 above
  sleep_hours: capped 0 below, 788 above
  stress_level: capped 0 below, 1580 above
  anxiety_score: capped 0 below, 1734 above
  digital_spending_per_month: capped 0 below, 9567 above
  ads_viewed_per_day: capped 329 below, 3436 above
  digital_addiction_score: capped 0 below, 1749 above
  wellbeing_index: capped 1160 below, 587 above
  productivity_score: capped 4639 below, 0 above

→ STEP 6 complete: Outlier treatment applied.


---
## STEP 7 — Feature Engineering (Derived KPI Columns)

**Rationale:** Create business-relevant derived columns that the Analysis and Visualization leads will use directly in Tableau and statistical notebooks.

| New Column | Formula | Business Use |
|-----------|---------|-------------|
| `productive_digital_ratio` | `education_content_hours / (social_media_hours + 0.01)` | Ratio of educational vs entertainment screen time |
| `sleep_risk_flag` | `1 if sleep_hours < 6 else 0` | Flag for sleep-deprived students |
| `high_stress_flag` | `1 if stress_level >= 7 else 0` | High-stress segment for wellbeing analysis |
| `total_screen_hours` | `online_learning_hours + social_media_hours` | Total daily screen engagement |
| `mental_health_composite` | `(anxiety_score + depression_score + stress_level) / 3` | Aggregated mental health risk score |

In [15]:
# --- KPI 1: Productive Digital Ratio ---
if all(c in df.columns for c in ['education_content_hours', 'social_media_hours']):
    df['productive_digital_ratio'] = (
        df['education_content_hours'] / (df['social_media_hours'] + 0.01)
    ).round(4)
    print('productive_digital_ratio created.')

# --- KPI 2: Sleep Risk Flag ---
if 'sleep_hours' in df.columns:
    df['sleep_risk_flag'] = (df['sleep_hours'] < 6).astype(int)
    print(f'sleep_risk_flag created. At-risk students: {df["sleep_risk_flag"].sum():,}')

# --- KPI 3: High Stress Flag ---
if 'stress_level' in df.columns:
    df['high_stress_flag'] = (df['stress_level'] >= 7).astype(int)
    print(f'high_stress_flag created. High-stress students: {df["high_stress_flag"].sum():,}')

# --- KPI 4: Total Screen Hours ---
if all(c in df.columns for c in ['online_learning_hours', 'social_media_hours']):
    df['total_screen_hours'] = (
        df['online_learning_hours'] + df['social_media_hours']
    ).round(3)
    print('total_screen_hours created.')

# --- KPI 5: Mental Health Composite Score ---
if all(c in df.columns for c in ['anxiety_score', 'depression_score', 'stress_level']):
    df['mental_health_composite'] = (
        (df['anxiety_score'] + df['depression_score'] + df['stress_level']) / 3
    ).round(4)
    print('mental_health_composite created.')

# --- KPI 6: Academic Risk Category (binned from academic_risk_score) ---
if 'academic_risk_score' in df.columns:
    df['academic_risk_category'] = pd.cut(
        df['academic_risk_score'],
        bins=[0, 33, 66, 100],
        labels=['Low Risk', 'Medium Risk', 'High Risk'],
        include_lowest=True
    )
    print('academic_risk_category created (binned into Low / Medium / High).')

print()
print('→ STEP 7 complete: Feature engineering done.')

productive_digital_ratio created.
sleep_risk_flag created. At-risk students: 246,142
high_stress_flag created. High-stress students: 39,610
total_screen_hours created.
mental_health_composite created.
academic_risk_category created (binned into Low / Medium / High).

→ STEP 7 complete: Feature engineering done.


---
## STEP 8 — Final Validation

Confirm the cleaned dataset is complete, correctly typed, and contains no unexpected nulls before saving.

In [16]:
print('=' * 60)
print('CLEANED DATASET — FINAL VALIDATION REPORT')
print('=' * 60)
print(f'Shape              : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Total missing vals : {df.isnull().sum().sum()}')
print(f'Duplicate rows     : {df.duplicated().sum()}')
print()
print('--- Column dtypes summary ---')
dtype_summary = df.dtypes.value_counts()
for dtype, count in dtype_summary.items():
    print(f'  {str(dtype):<20} : {count} columns')
print()
print('--- Engineered KPI columns preview ---')
kpi_cols = [
    'productive_digital_ratio', 'sleep_risk_flag', 'high_stress_flag',
    'total_screen_hours', 'mental_health_composite', 'academic_risk_category'
]
available_kpi = [c for c in kpi_cols if c in df.columns]
print(df[available_kpi].describe().to_string())
print('=' * 60)

CLEANED DATASET — FINAL VALIDATION REPORT
Shape              : 500,000 rows × 57 columns
Total missing vals : 103890
Duplicate rows     : 0

--- Column dtypes summary ---
  float64              : 38 columns
  int64                : 6 columns
  category             : 2 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns
  category             : 1 columns

--- Engineered KPI columns preview ---
       productive_digital_ratio  sleep_risk_flag  high_stress_flag  total_screen_hours  mental_health_composite
count               500000.0000      500000.0000       500000.0000         500000.0000              500000.0000
mean                     0.1660           0.4923            0.0792        

In [17]:
# Preview cleaned data
df.head(5)

,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,device_access,internet_access_hours,education_level,field_of_study,academic_motivation,online_learning_hours,social_media_hours,sessions_per_day,average_session_length_minutes,late_night_usage,education_content_hours,short_video_hours,entertainment_content_hours,news_content_hours,likes_given_per_day,comments_written_per_day,posts_created_per_week,late_night_score,brain_rot_index,brain_rot_level,attention_span_minutes,study_hours_per_week,class_attendance_rate,productivity_score,sleep_hours,stress_level,anxiety_score,depression_score,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score,late_night_usage_encoded,cyberbullying_exposure_flag,adult_content_exposure_flag,brain_rot_level_encoded,productive_digital_ratio,sleep_risk_flag,high_stress_flag,total_screen_hours,mental_health_composite,academic_risk_category
student_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,Qatar,Developing,16.3000,54.9300,27.1900,21,Male,Rural,High,Smartphone,5.2395,Graduate,Business,7.1772,10.4244,1.0987,4.6488,16.0574,Sometimes,0.2399,0.4570,0.4018,0.3294,30.0963,4.9677,2.1084,1,5.8685,Low,57.5522,31.4342,95.4824,10.0000,7.1390,4.2620,3.1929,1.4091,38.6995,3.1196,3.6811,79.5082,No,No,8.1799,66.6622,0.0000,26.5167,1.0000,0,0,1.0000,0.2164,0,0,11.5230,2.9547,Low Risk
2,Usa,Developed,8.7500,94.3900,85.3400,25,Male,Urban,Middle,Smartphone,6.2624,Diploma,Arts,5.4917,8.1464,5.6115,15.1404,63.0832,Often,0.3832,3.3061,1.9222,0.6437,114.3857,3.1745,5.4780,2,29.5948,Low,51.3099,25.9784,86.6170,7.1794,4.0000,8.2135,8.0649,7.6952,157.4004,18.3581,6.5389,73.4525,No,Yes,22.0731,43.8923,0.0000,39.2577,2.0000,0,1,1.0000,0.0682,1,1,13.7580,7.9912,Low Risk
3,Mexico,Developing,23.6400,47.2400,73.5500,18,Female,Urban,Low,Smartphone,3.4377,Dropout,Not Applicable,5.3950,5.1545,2.1999,7.0752,25.7789,Sometimes,0.3954,0.8847,0.9198,0.4122,56.3332,8.2406,1.1245,1,20.2423,Low,55.8444,21.4475,89.6282,9.5730,6.8093,2.9313,1.7192,1.0000,79.6035,11.7583,5.1507,35.7531,No,No,12.5918,65.4846,0.0000,47.5422,1.0000,0,0,1.0000,0.1789,0,0,7.3540,1.8835,Low Risk
4,Canada,Developed,14.5100,90.5000,188.1900,25,Male,Urban,Middle,Laptop,4.2037,Phd,Social Sciences,7.6189,8.6742,2.2520,8.3423,28.2821,Never,0.8601,1.2004,0.1915,0.8097,41.7379,7.6891,1.8241,0,7.3578,Low,58.0295,31.1823,93.3856,10.0000,7.5372,4.3561,4.8608,1.7552,69.3183,7.9062,3.1954,47.6075,No,Yes,8.0082,57.9094,0.0000,23.4361,0.0000,0,1,1.0000,0.3802,0,0,10.9260,3.6574,Low Risk
5,Sri Lanka,Underdeveloped,62.2800,36.8400,11.0200,15,Other,Rural,Middle,Smartphone,5.4191,School,Not Applicable,5.4707,8.3970,5.3759,15.4916,58.3893,Sometimes,1.0571,1.8846,2.4341,0.5692,73.5232,16.1315,9.8150,1,22.3477,Low,60.0000,23.2907,92.6183,9.3375,5.5222,6.5786,6.0255,3.6786,144.0199,19.4278,7.1802,82.9413,No,No,21.5513,53.6864,0.0000,47.3085,1.0000,0,0,1.0000,0.1963,1,0,13.7730,5.4276,Low Risk


---
## STEP 9 — Save Cleaned Dataset to `data/processed/`

In [18]:
df.to_csv(PROCESSED_PATH, index=True)  # keep student_id as index column in file

# Verify saved file
saved = pd.read_csv(PROCESSED_PATH, index_col='student_id', nrows=3)
print(f'✅ Cleaned dataset saved to: {PROCESSED_PATH}')
print(f'   File size: {os.path.getsize(PROCESSED_PATH) / 1024:.1f} KB')
print(f'   Columns saved: {saved.shape[1]}')
print()
print('─' * 60)
print('CLEANING PIPELINE COMPLETE')
print('─' * 60)
print('Steps executed:')
steps = [
    'student_id set as index',
    'Missing values filled (field_of_study → Not Applicable; numerics → median)',
    'String columns stripped and title-cased',
    'Categorical encoding: late_night_usage, cyberbullying, adult_content, brain_rot_level',
    'Data types corrected (category, int)',
    'Outliers Winsorised at 1.5×IQR fences',
    'Feature engineering: 6 new KPI columns',
    'Final validation passed',
    f'Output saved → {PROCESSED_PATH}'
]
for i, s in enumerate(steps, 1):
    print(f'  {i}. {s}')
print()
print('→ Next step: Run 03_eda.ipynb')

✅ Cleaned dataset saved to: ../data/processed/student_digital_behaviour_cleaned.csv
   File size: 334687.4 KB
   Columns saved: 57

────────────────────────────────────────────────────────────
CLEANING PIPELINE COMPLETE
────────────────────────────────────────────────────────────
Steps executed:
  1. student_id set as index
  2. Missing values filled (field_of_study → Not Applicable; numerics → median)
  3. String columns stripped and title-cased
  4. Categorical encoding: late_night_usage, cyberbullying, adult_content, brain_rot_level
  5. Data types corrected (category, int)
  6. Outliers Winsorised at 1.5×IQR fences
  7. Feature engineering: 6 new KPI columns
  8. Final validation passed
  9. Output saved → ../data/processed/student_digital_behaviour_cleaned.csv

→ Next step: Run 03_eda.ipynb


---
## Cleaning Log Summary

*(Fill in after running the notebook — ETL Lead to complete)*

| Step | Column(s) | Transformation Applied | Rows / Values Affected |
|------|-----------|----------------------|------------------------|
| 1 | `student_id` | Set as DataFrame index | — |
| 2 | `field_of_study` | NaN → 'Not Applicable' | *fill in count* |
| 2 | Numeric cols | NaN → column median | *fill in count* |
| 3 | All string cols | `.strip().title()` | *fill in if cardinality changed* |
| 4 | `late_night_usage` | Ordinal: Never=0, Sometimes=1, Often=2 | All rows |
| 4 | `cyberbullying_exposure` | Binary: No=0, Yes=1 | All rows |
| 4 | `adult_content_exposure` | Binary: No=0, Yes=1 | All rows |
| 4 | `brain_rot_level` | Ordinal: Low=1, Medium=2, High=3 | All rows |
| 5 | 12 columns | Cast to `category` dtype | All rows |
| 6 | 13 numeric cols | Winsorised at 1.5×IQR | *fill in count* |
| 7 | — | 6 new KPI columns derived | All rows |